# 01 — Baseline models (Step 3)

Runs the three **un-fine-tuned** models — Llama 3.1 8B, Qwen 3.5 9B, Mistral 7B — on emails from our dataset, using the shared prompt in `src/prompts.py`, and saves every answer to Google Drive.

**Start small:** the default settings run **20 validation emails** per model, to check that every model loads and answers in the right JSON format. Once that works, change the config cell to the full test set (see the end of the notebook).

### Before you run
1. **GPU runtime:** *Runtime → Change runtime type → GPU* (T4 works; L4/A100 is much faster).
2. **Hugging Face token:** create a *read* token at https://huggingface.co/settings/tokens, then in Colab click the 🔑 *Secrets* icon in the left sidebar, add a secret named `HF_TOKEN`, and enable notebook access.
3. **Llama license:** Llama 3.1 is gated. Open https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct while logged in and accept the license (approval usually takes minutes to an hour).

## 1. Setup — check the GPU, get the code, install libraries

In [ ]:
# Show which GPU Colab gave us. If this fails, the runtime has no GPU —
# change it under Runtime → Change runtime type.
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os, sys

# Let PyTorch grow its GPU memory pool in flexible chunks. This reduces
# "fragmentation" — free memory split into pieces too small to use — which
# the out-of-memory error message itself recommends. It must be set BEFORE
# torch is imported, which is why it is in this first cell.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Get our project code from GitHub. If it is already there (re-running the
# cell), just pull the latest version instead of cloning again.
REPO_DIR = "/content/llm-phishing"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q https://github.com/mihaighise/llm-phishing.git {REPO_DIR}

# Install / upgrade the libraries for running models:
#   transformers  — loads the models and their chat templates
#                   (must be recent: Qwen 3.5 is a new architecture)
#   accelerate    — places the model on the GPU (device_map="auto")
#   bitsandbytes  — 4-bit quantization so 8-9B models fit in GPU memory
!pip install -q -U transformers accelerate bitsandbytes

# OPTIONAL SPEED-UP FOR QWEN 3.5.
# Qwen 3.5 uses a new kind of attention layer ("gated delta rule"). Without
# the `flash-linear-attention` (fla) library, transformers runs a slow
# reference version of it (the yellow "falling back to its reference PyTorch
# implementation" warnings) — about 30 s per email on a T4.
# fla is built for newer GPUs, so we only install it when the GPU's
# "compute capability" is 8.0 or higher (L4 = 8.9, A100 = 8.0, T4 = 7.5).
# On a T4 we skip it and keep the slow-but-working fallback.
# This changes speed only, not the answers.
import torch
major, minor = torch.cuda.get_device_capability()
if major >= 8:
    print(f"GPU compute capability {major}.{minor}: installing flash-linear-attention")
    !pip install -q flash-linear-attention
    try:
        import fla  # check that it installed and imports cleanly
        print("flash-linear-attention ready — Qwen will use the fast layers")
    except Exception as exc:
        # If it is broken, remove it so transformers safely falls back to the
        # slow version instead of crashing during the Qwen run.
        print("flash-linear-attention failed to import, removing it:", exc)
        !pip uninstall -q -y flash-linear-attention
else:
    print(f"GPU compute capability {major}.{minor}: skipping flash-linear-attention "
          "(Qwen will run, but slowly)")

# Make `import prompts` / `import inference` work by adding src/ to the
# list of folders Python searches for modules.
sys.path.insert(0, f"{REPO_DIR}/src")

## 2. Log in to Hugging Face
Needed to download the gated Llama model. Reads the `HF_TOKEN` secret you added.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# userdata.get reads a Colab secret, so the token never appears in the
# notebook itself (important if you share or commit the notebook).
login(token=userdata.get("HF_TOKEN"))

## 3. Build the dataset on Colab
The processed data is not in GitHub (it is gitignored), so we rebuild it here: download the 5 Zenodo files (~60 MB) and run the same `build_dataset.py`. The build is deterministic, so the result is **identical** to the files on your Mac — the fingerprint check below proves it.

In [ ]:
import hashlib
from pathlib import Path

RAW_DIR = Path(REPO_DIR) / "data" / "zenodo"
DATA_DIR = Path(REPO_DIR) / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Download each source CSV unless it is already present.
for name in ["Nazario", "Nigerian_Fraud", "Nazario_5", "Nigerian_5", "SpamAssasin"]:
    target = RAW_DIR / f"{name}.csv"
    if not target.exists():
        url = f"https://zenodo.org/api/records/8339691/files/{name}.csv/content"
        !curl -sfL -o {target} "{url}"

# Build the splits (same command you run locally).
!python3 {REPO_DIR}/src/build_dataset.py

# Fingerprints (MD5 hashes) of the files built on the Mac. If these match,
# Colab is evaluating exactly the same emails.
EXPECTED = {
    "test.jsonl": "50b428d99f00fc8bf320d827acd4aefc",
    "val.jsonl":  "8e4da4755a9e99c67e6d43d7f2ef7fc3",
}
for fname, expected in EXPECTED.items():
    actual = hashlib.md5((DATA_DIR / fname).read_bytes()).hexdigest()
    status = "OK — identical to local build" if actual == expected else \
             "MISMATCH — data differs from the local build!"
    print(f"{fname}: {status}")

## 4. Save results to Google Drive
Colab deletes its files when the session ends, so results go to your Drive. If the session disconnects mid-run, re-running picks up where it stopped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RESULTS_DIR = Path("/content/drive/MyDrive/llm-phishing/results/raw")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Results will be saved in:", RESULTS_DIR)

## 5. Configuration
Everything you might want to change is in this one cell.

In [ ]:
# Make sure Python can find our modules in src/ (needed for the
# PROMPT_VERSION import below, also after a runtime restart).
import sys
SRC_DIR = "/content/llm-phishing/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import json
from prompts import PROMPT_VERSION

SPLIT = "val"        # "val" for testing the setup, "test" for the real results
N_EMAILS = 20        # how many emails to run; None = the whole split
BATCH_SIZE = 8       # emails processed at once; lower to 4 if you get "CUDA out of memory"
MODELS_TO_RUN = ["llama31_8b", "qwen35_9b", "mistral_7b"]

# Load the chosen split: one email (a dict) per line.
with open(DATA_DIR / f"{SPLIT}.jsonl", encoding="utf-8") as f:
    all_emails = [json.loads(line) for line in f]

if N_EMAILS is None:
    emails = all_emails
else:
    # Take a BALANCED sample: half phishing, half legitimate. The split file
    # is already shuffled, so "the first N/2 of each class" is random.
    half = N_EMAILS // 2
    phish = [e for e in all_emails if e["label"] == "phishing"][:half]
    legit = [e for e in all_emails if e["label"] == "legitimate"][:N_EMAILS - half]
    emails = phish + legit

# The result file name records what was run, e.g. "llama31_8b_val_20_v2.jsonl".
# PROMPT_VERSION is part of the name, so results from an older prompt are
# never skipped-as-done or mixed with results from the current prompt.
RUN_TAG = f"{SPLIT}_{'all' if N_EMAILS is None else N_EMAILS}_{PROMPT_VERSION}"
print(f"Prompt {PROMPT_VERSION}: running {len(emails)} emails from '{SPLIT}' on {MODELS_TO_RUN}")

Preview exactly what a model receives for one email (system prompt + user message):

In [ ]:
# Make sure Python can find our modules in src/. Cell 1 already does this,
# but if Colab restarts the runtime (crash, disconnect, "Restart session"),
# that setting is forgotten and imports fail with "No module named ...".
# Re-adding it here makes this cell safe to run on its own after a restart.
import sys
SRC_DIR = "/content/llm-phishing/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from prompts import build_messages

for message in build_messages(emails[0]):
    print(f"===== {message['role'].upper()} =====")
    print(message["content"][:2500])

## 6. Run the models
Each model is downloaded (~5-6 GB each, first time only per session), loaded in 4-bit, run on the emails, then removed from GPU memory before the next one loads.

Rough speed per email: T4 ≈ 3-6 s, L4 ≈ 1-2 s, A100 < 1 s. On a T4, Qwen 3.5 is much slower (~30 s per email) because the fast version of its new attention layers (`flash-linear-attention`) is only installed on L4/A100 GPUs (see the setup cell).

In [ ]:
# Make sure Python can find our modules in src/. Cell 1 already does this,
# but if Colab restarts the runtime (crash, disconnect, "Restart session"),
# that setting is forgotten and imports fail with "No module named ...".
# Re-adding it here makes this cell safe to run on its own after a restart.
import sys
SRC_DIR = "/content/llm-phishing/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from inference import load_model, free_gpu_memory, run_model

# If this cell crashed earlier, a model may still be sitting in GPU memory.
# Remove it before loading anything new.
model = tokenizer = None
free_gpu_memory()

all_results = {}
for name in MODELS_TO_RUN:
    print(f"\n=== {name} ===")
    model, tokenizer = load_model(name)
    out_path = RESULTS_DIR / f"{name}_{RUN_TAG}.jsonl"
    all_results[name] = run_model(name, model, tokenizer, emails, out_path,
                                  batch_size=BATCH_SIZE)
    # Free the GPU for the next model. `del` removes the notebook's own
    # references; only then can free_gpu_memory() actually release the memory.
    # (Without this, the next model loads ON TOP of this one and a 15 GB T4
    # runs out of memory.)
    del model, tokenizer
    free_gpu_memory()

## 7. Quick check
For the 20-email smoke test, the most important column is **parse rate** — every model should be close to 100%. Accuracy on 20 emails is only a rough hint; the real metrics come in Step 6 on the full test set.

In [ ]:
import pandas as pd

rows = []
for name, results in all_results.items():
    df = pd.DataFrame(results)
    rows.append({
        "model": name,
        "emails": len(df),
        # Share of answers that were valid JSON with a valid label.
        "parse_rate": df["parse_ok"].mean(),
        # Unparseable answers count as wrong (label None != true label).
        "accuracy": (df["label"] == df["true_label"]).mean(),
        "avg_indicators": df["indicators"].apply(len).mean(),
    })
pd.DataFrame(rows).round(3)

Look at a few raw answers — the best way to spot format problems early:

In [ ]:
for name, results in all_results.items():
    print(f"\n######## {name} ########")
    for r in results[:2]:
        print(f"--- true: {r['true_label']} | parsed: {r['label']} | ok: {r['parse_ok']} | error: {r['error']}")
        print(r["raw_output"][:600])

## Next: the full baseline run
When all three models parse close to 100% on the 20-email check:
1. In the configuration cell set `SPLIT = "test"` and `N_EMAILS = None` (all 1,000 emails).
2. Run all cells again. Each model takes roughly 15-30 min on an L4/A100, 1-1.5 h on a T4.
3. Results land in `MyDrive/llm-phishing/results/raw/<model>_test_all_<prompt version>.jsonl`, ready for the metrics in Step 6.

**Do not** tweak the prompt based on test-set results — only on the validation set. Otherwise the test set is no longer "unseen" and the comparison is biased.